# GSEA — Clinical Outcome (Good vs Poor), single-cell pseudobulk

* **What this does.** Runs preranked GSEA on the single-cell pseudobulk pydeseq2 results **and** draws the ridge plots, in one notebook.
* **Input (read from disk).** `${ACC_DATA_ROOT}/outputs/csv_files/pydeseq2/clinical_outcome/DEG_results.csv` — first column is the gene symbol index named `gene`, then `baseMean`, `log2FoldChange`, `lfcSE`, `stat`, `pvalue`, `padj` (~9,867 rows). Nothing here depends on any in-memory object from the DE notebook.
* **Databases — three.** `MSigDB_C5_BP.gmt` (GO Biological Process, db key `BP`), `MSigDB_C2_Reactome.gmt` (Reactome, `reactome`) and `MSigDB_Hallmark.gmt` (`hallmark`). No CC / MF / KEGG. **C6 oncogenic signatures are deliberately disabled** — the entry is commented out in `GMT_MAP`, so that database is neither run nor plotted.
* **Ranking metric.** The raw Wald `stat` from pydeseq2, sorted descending. **Direction convention:** positive `stat` / positive NES = **higher in Good**; negative `stat` / negative NES = **higher in Poor**.
* **Figures — ridge plots only.** One figure per database, written as SVG to `outputs/images/gsea/clinical_outcome/ridge_plots/`. The NES dot plots this notebook used to draw have been retired.
* **What a ridge is.** For each selected term, the ridge is a Gaussian KDE of the **ranking-metric values of that term's leading-edge genes**, evaluated on one shared x-grid and peak-normalised to 1.0 — so height encodes *shape*, never gene count (the count is annotated instead). Each figure shows the **top 3 terms per direction** as up to 3 paired rows: the i-th strongest Poor term left of zero, the i-th strongest Good term right of zero, sharing a baseline. Pairing is by rank within a direction, not by any relationship between the two terms.
* **Style.** Ported from `bulk_rna_data/scripts/named_scripts/gsea_ridgeplot_clinical_outcome.ipynb` so the single-cell panels are visually identical to the bulk ridge plots.
* **Selection.** fdr < 0.05, then fdr ascending with |NES| descending as the tie-break. When a direction has fewer than 3 significant terms the figure simply shows fewer ridges — no padding from the other direction, no relaxed cutoff.
* **Caveat — enrichment background.** The single-cell DE universe is a **10,000-gene HVG panel**, so enrichment here is relative to an **HVG background**, not to the whole transcriptome. Terms are effectively tested against the genes that survived highly-variable-gene selection, which biases the universe toward variable genes; read the absolute FDRs with that in mind.
* **Comparability with bulk.** GSEA parameters deliberately mirror the bulk script — **seed 6, 1,000 permutations, `min_size` 15**, fdr < 0.05 — so the single-cell and bulk pathway results are directly comparable.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# necessary packages
import os
import time
import logging
import textwrap

import numpy as np
import pandas as pd
import gseapy as gp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.transforms as mtransforms
import seaborn as sns
from scipy.stats import gaussian_kde
from numpy.linalg import LinAlgError

## Configuration

All paths absolute. GSEA parameters are pinned to the bulk script's values so the two runs stay comparable; `max_size` is gseapy's default (500) and is stated explicitly here rather than left implicit. Ridge geometry and colours are pinned to the bulk ridge-plot notebook for the same reason.

In [ ]:
# ---- paths -------------------------------------------------------------------
DEG_CSV   = f'{ACC_DATA_ROOT}/outputs/csv_files/pydeseq2/clinical_outcome/DEG_results.csv'
GMT_DIR   = f'{ACC_DATA_ROOT}/reference/msigdb_gmt'
GSEA_OUT  = f'{ACC_DATA_ROOT}/outputs/gsea/clinical_outcome'
RIDGE_DIR = f'{ACC_DATA_ROOT}/outputs/images/gsea/clinical_outcome/ridge_plots'

# ---- databases: gmt file -> (db key, output csv) -----------------------------
GMT_MAP = {
    'MSigDB_C5_BP.gmt':            ('BP',             'MSigDB_C5_BP_clinical_outcome.csv'),
    'MSigDB_C2_Reactome.gmt':      ('reactome',       'MSigDB_C2_Reactome_clinical_outcome.csv'),
    'MSigDB_Hallmark.gmt':         ('hallmark',       'MSigDB_Hallmark_clinical_outcome.csv'),
    # DISABLED - C6 oncogenic signatures are deliberately not run or plotted. Everything
    # downstream (the prerank loop, the ridge plots, the verification cell) is driven off
    # GMT_MAP, so leaving this line commented is the whole switch. Uncomment to bring it back.
    # 'MSigDB_C6_Oncogenic_sigs.gmt': ('oncogenic_sigs', 'MSigDB_C6_Oncogenic_sigs_clinical_outcome.csv'),
}
DB_ORDER  = [db for db, _ in GMT_MAP.values()]
CSV_BY_DB = {db: csv_name for db, csv_name in GMT_MAP.values()}

# pretty names for figure titles
DB_TITLES = {
    'BP':             'GO Biological Process',
    'reactome':       'Reactome',
    'hallmark':       'MSigDB Hallmark',
    # 'oncogenic_sigs': 'Oncogenic Signatures',   # disabled, see GMT_MAP above
}

# ---- GSEA parameters (mirror the bulk GSEA script exactly) -------------------
SEED            = 6
PERMUTATION_NUM = 1000    # also sets the fdr resolution floor used by fdr_label()
MIN_SIZE        = 15
MAX_SIZE        = 500     # gseapy default; the bulk script omitted it, we state it explicitly
THREADS         = 8

# ---- selection parameters ----------------------------------------------------
FDR_CUTOFF = 0.05    # stricter than the usual GSEA 0.25 - deliberate
TOP_N      = 3       # terms per direction, per database -> TOP_N paired rows

# ---- ridge geometry ---------------------------------------------------------
# Ridge overlap. Every ridge is peak-normalised to the same height and the fills are opaque,
# so whatever fraction sits above the next baseline is hidden behind the row above - on EVERY
# row at once. Large overlaps flat-top every ridge and swallow real structure; 0.12 keeps the
# ridgeline interlock while leaving every visible mode intact.
OVERLAP     = 0.12   # fraction of a ridge's height that the row above sits over
GRID_N      = 512    # x resolution of every KDE
SUPPORT_EPS = 0.003  # densities below this fraction of a ridge's peak are not drawn
X_PAD       = 0.03   # fraction of the ranking range added as padding on each side

# ---- colours ----------------------------------------------------------------
# fixed project brand colours for the clinical-outcome contrast
GOOD_COLOR = '#2c9d2a'   # positive NES -> higher in Good -> right of zero
POOR_COLOR = '#E63946'   # negative NES -> higher in Poor -> left of zero
DIR_COLORS = {'Higher in Good': GOOD_COLOR, 'Higher in Poor': POOR_COLOR}
DIR_EDGES  = {'Higher in Good': '#1d6b1c', 'Higher in Poor': '#a6202b'}   # darker outlines

# prefixes stripped from term names for display.
# C6 oncogenic terms carry no database prefix (e.g. 'AKT_UP.V1_DN'), so they pass through
# unchanged apart from the shared de-underscore / title-case pass.
DB_PREFIXES = ('HALLMARK_', 'GOBP_', 'GOCC_', 'GOMF_', 'REACTOME_', 'KEGG_MEDICUS_')

# MSigDB term names are SHOUTED, so display uses .title() - which mangles gene symbols and
# nucleic-acid acronyms ('RRNA' -> 'Rrna', 'MYC' -> 'Myc', 'RTK PLCG ITPR' -> 'Rtk Plcg Itpr').
# This maps the title-cased form back to the correct casing, word by word.
ACRONYMS = {
    'Rrna': 'rRNA', 'Mrna': 'mRNA', 'Trna': 'tRNA', 'Snrna': 'snRNA', 'Snorna': 'snoRNA',
    'Ncrna': 'ncRNA', 'Dna': 'DNA', 'Rna': 'RNA', 'Atp': 'ATP', 'Gtp': 'GTP', 'Ecm': 'ECM',
    'Mhc': 'MHC', 'Er': 'ER', 'Uv': 'UV', 'Il': 'IL', 'Mtor': 'mTOR', 'Nfkb': 'NF-kB',
    'Tgf': 'TGF', 'Tnf': 'TNF', 'Myc': 'MYC', 'Akt': 'AKT', 'Atf2': 'ATF2', 'Vegf': 'VEGF',
    'Shh': 'SHH', 'Gcnp': 'GCNP', 'Rtk': 'RTK', 'Plcg': 'PLCG', 'Itpr': 'ITPR',
    'Kras': 'KRAS', 'Egfr': 'EGFR', 'Pten': 'PTEN', 'Tp53': 'TP53', 'Raf': 'RAF',
    'Mek': 'MEK', 'Src': 'SRC', 'Ic': 'IC', 'Sumoylation': 'SUMOylation',
    'Esc': 'ESC', 'Stk33': 'STK33',
}
LABEL_MAXLEN = 62   # truncate very long term labels
LABEL_WRAP   = 28   # wrap width for the term labels flanking the axes

os.makedirs(GSEA_OUT, exist_ok=True)
os.makedirs(RIDGE_DIR, exist_ok=True)

# ---- inputs must exist before anything runs ----------------------------------
assert os.path.exists(DEG_CSV), DEG_CSV
for gmt_file in GMT_MAP:
    assert os.path.exists(os.path.join(GMT_DIR, gmt_file)), gmt_file

# ---- fonts -------------------------------------------------------------------
# Arial is NOT installed here. Liberation Sans is, and is metrically identical to
# Arial (same advance widths). Listing both means:
#   * matplotlib resolves metrics from LiberationSans-Regular.ttf -> Arial-correct layout
#   * svg.fonttype='none' writes the stack "'Arial', 'Liberation Sans'" into the SVG,
#     so Illustrator/journals with Arial render true Arial, others fall back cleanly.
# Setting font.family='Arial' alone would silently lay out with DejaVu Sans (~8% wider).
plt.rcParams['font.family'] = ['Arial', 'Liberation Sans']
plt.rcParams['svg.fonttype'] = 'none'      # keep SVG text editable in Illustrator/Inkscape

# 'Arial' is deliberately first so it lands in the SVG's font-family, but since it is not
# installed here font_manager logs "findfont: Font family 'Arial' not found" for EVERY text
# object (~200 lines/run). The lookup then falls through to Liberation Sans, which is exactly
# what we want, so the message is noise rather than a problem - silence it.
# (This is a logging call, not a warning, so warnings.filterwarnings does not catch it.)
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

# ---- publication font sizes --------------------------------------------------
plt.rcParams['font.size']            = 14
plt.rcParams['axes.titlesize']       = 16
plt.rcParams['axes.labelsize']       = 15
plt.rcParams['xtick.labelsize']      = 13
plt.rcParams['ytick.labelsize']      = 13
plt.rcParams['legend.fontsize']      = 13
plt.rcParams['legend.title_fontsize'] = 13

print(f'deg csv   : {DEG_CSV}')
print(f'gmt dir   : {GMT_DIR}')
print(f'gsea out  : {GSEA_OUT}')
print(f'ridge out : {RIDGE_DIR}')
print(f'databases : {DB_ORDER}')
print(f'params    : seed={SEED}, perms={PERMUTATION_NUM}, min_size={MIN_SIZE}, '
      f'max_size={MAX_SIZE}, threads={THREADS}, fdr<{FDR_CUTOFF}, top {TOP_N}/direction')
print(f'font      : {plt.rcParams["font.family"]}')

## GSEA helpers

Copied verbatim from `bulk_rna_data/scripts/named_scripts/gsea_clinical_outcome.ipynb` so the two runs produce byte-compatible CSV schemas.

In [ ]:
def gmt_sizes(gmt_path):
    """Map term -> number of genes in the raw GMT (gseapy only reports the matched size)."""
    sizes = {}
    with open(gmt_path) as fh:
        for line in fh:
            fields = line.rstrip('\n').split('\t')
            if len(fields) > 2:
                sizes[fields[0]] = len([g for g in fields[2:] if g])
    return sizes


def prerank_to_df(pre_res, sizes):
    """Flatten pre_res.results into a tidy DataFrame.

    Term/fdr/es/nes come first and keep their original names so the existing
    bar_plot_gsea_outputs.ipynb reads these files unchanged.
    """
    out = []
    for term, res in pre_res.results.items():
        # gseapy reports 'tag %' as a "lead/matched" string, e.g. "51/144"
        lead_size, matched_size = (int(x) for x in res['tag %'].split('/'))
        out.append([
            term,
            res['fdr'],
            res['es'],
            res['nes'],
            res['pval'],
            sizes.get(term, np.nan),
            matched_size,
            lead_size / matched_size if matched_size else np.nan,
            res['lead_genes'],
        ])

    return (pd.DataFrame(out, columns=['Term', 'fdr', 'es', 'nes', 'pval',
                                       'geneset_size', 'matched_size', 'tag_pct', 'lead_genes'])
              .sort_values('fdr')
              .reset_index(drop=True))

## Build the ranking

Genes are ranked by the raw Wald `stat`, descending: the top of the list is **higher in Good**, the bottom is **higher in Poor**.

The `dropna` / `astype(str)` / `drop_duplicates` guards are the documented schema contract — in practice `stat` has no NaN and gene symbols are already unique, so they are no-ops, but they cost nothing and keep the cell safe if the DE step changes.

`ranking` is the same table as a `gene -> stat` Series. The ridge plots look leading-edge genes up in it, and because gseapy writes `lead_genes` in exactly the gene names it was handed in `rnk`, that lookup is total by construction — no symbol-namespace join is needed here (the bulk notebook needs one; this one does not).

> **Expected log noise.** gseapy will print `Duplicated values found in preranked stats: ~1% of genes`. That is expected: a small number of genes share near-zero `stat` values, so their relative order inside the tie block is arbitrary. With ties this rare and this close to the middle of the ranking, the effect on enrichment scores is negligible — harmless, not an error.

In [ ]:
res = pd.read_csv(DEG_CSV)
res = res.rename(columns={res.columns[0]: 'gene'})

rnk = (res.dropna(subset=['stat'])[['gene', 'stat']]
          .assign(gene=lambda d: d['gene'].astype(str))
          .drop_duplicates('gene')
          .sort_values('stat', ascending=False)
          .reset_index(drop=True))

# same ranking, keyed for the leading-edge lookups the ridge plots do
ranking = rnk.set_index('gene')['stat'].astype(float)

print(f'DEG table : {res.shape[0]} rows x {res.shape[1]} cols')
print(f'columns   : {list(res.columns)}')
print(f'ranked    : {len(rnk)} genes, stat from {ranking.min():.3f} to {ranking.max():.3f}\n')

print('--- top 10 (higher in Good) ---')
print(rnk.head(10).to_string(index=False))
print('\n--- bottom 10 (higher in Poor) ---')
print(rnk.tail(10).to_string(index=False))

## Run GSEA

`outdir=None` and `no_plot=True` stop gseapy from scattering its own report directories and per-term plots across the project — the only artefacts we want are the two CSVs written here.

CSVs are written **with** the pandas index, exactly as the bulk script does, so downstream readers behave identically (the `Unnamed: 0` column is dropped again at plot time).

In [ ]:
results = {}

for gmt_file, (db, csv_name) in GMT_MAP.items():
    gmt_path = os.path.join(GMT_DIR, gmt_file)
    t0 = time.time()

    pre = gp.prerank(rnk=rnk,
                     gene_sets=gmt_path,
                     outdir=None,
                     no_plot=True,
                     min_size=MIN_SIZE,
                     max_size=MAX_SIZE,
                     permutation_num=PERMUTATION_NUM,
                     threads=THREADS,
                     seed=SEED)

    out_df = prerank_to_df(pre, gmt_sizes(gmt_path))
    out_df.to_csv(os.path.join(GSEA_OUT, csv_name))   # with the pandas index, same as bulk
    results[db] = out_df

    sig = out_df[out_df['fdr'] < FDR_CUTOFF]
    print(f'{gmt_file:24s} -> {csv_name}')
    print(f'  {len(out_df):5d} terms tested | {len(sig):4d} at fdr<{FDR_CUTOFF} '
          f'| {int((sig["nes"] > 0).sum()):4d} Good (nes>0) '
          f'| {int((sig["nes"] < 0).sum()):4d} Poor (nes<0) '
          f'| {time.time() - t0:6.1f}s')

## Summary across the databases

`sig_up_Good` = significant terms with NES > 0 (**higher in Good**); `sig_dn_Poor` = significant terms with NES < 0 (**higher in Poor**).

In [ ]:
summary = []
for gmt_file, (db, csv_name) in GMT_MAP.items():
    out_df = results[db]
    sig = out_df[out_df['fdr'] < FDR_CUTOFF]
    summary.append([csv_name,
                    len(out_df),
                    len(sig),
                    int((sig['nes'] > 0).sum()),
                    int((sig['nes'] < 0).sum())])

summary_df = pd.DataFrame(summary, columns=['file', 'terms_tested',
                                            f'sig_fdr<{FDR_CUTOFF}', 'sig_up_Good', 'sig_dn_Poor'])
print(summary_df.to_string(index=False))
summary_df

In [ ]:
# top significant terms per database, for a quick look
for db in DB_ORDER:
    out_df = results[db]
    sig = out_df[out_df['fdr'] < FDR_CUTOFF]
    print(f'\n=== {db} — {len(sig)} significant terms at fdr < {FDR_CUTOFF} ===')
    if len(sig):
        print(sig[['Term', 'nes', 'fdr', 'matched_size']].head(10).to_string(index=False))
    else:
        print('  (none)')

## Ridge-plot helpers

Ported from `bulk_rna_data/scripts/named_scripts/gsea_ridgeplot_clinical_outcome.ipynb` so the single-cell panels are visually identical to the bulk ones. The one deliberate simplification: `lead_stats` reads straight from `ranking`, since the single-cell `lead_genes` are already in the same gene-symbol namespace.

In [ ]:
def clean_term(term, maxlen=LABEL_MAXLEN, wrap=LABEL_WRAP):
    """Strip the MSigDB prefix, de-underscore, restore acronyms, truncate and wrap a term."""
    label = str(term)
    for prefix in DB_PREFIXES:
        if label.startswith(prefix):
            label = label[len(prefix):]
            break
    label = label.replace('_', ' ').strip().title()
    label = ' '.join(ACRONYMS.get(w, w) for w in label.split(' '))
    if maxlen is not None and len(label) > maxlen:
        label = label[:maxlen - 1].rstrip() + '…'
    if wrap:
        label = '\n'.join(textwrap.wrap(label, wrap)) or label
    return label


def fdr_label(fdr):
    """Format an FDR for annotation.

    gseapy reports fdr == 0 whenever no permutation beat the observed score. With
    PERMUTATION_NUM = 1000 that is a resolution floor, NOT infinite significance, so it is
    reported as an inequality rather than as a literal zero.
    """
    if fdr <= 0:
        return f'FDR < {1 / PERMUTATION_NUM:g}'
    return f'FDR = {fdr:.3g}'


def select_top(sig, n_per_direction):
    """Top n terms per direction: fdr ascending, ties broken by |NES| descending.

    The tie-break is not cosmetic. Hundreds of terms share fdr == 0 (reactome: 239
    of them), so ranking on fdr alone would return an arbitrary slice of the tie
    block - "whichever came first in the CSV" rather than the strongest terms.
    """
    return (sig.assign(abs_nes=sig['nes'].abs())
               .sort_values(['fdr', 'abs_nes'], ascending=[True, False])
               .groupby('direction', group_keys=False)
               .head(n_per_direction)
               .sort_values('nes'))


def lead_stats(lead_genes, ranking):
    """Ranking-metric values of one term's leading-edge genes.

    Asserts full recovery: gseapy writes lead_genes in the gene names it was handed in `rnk`,
    which is the same index `ranking` carries, so anything missing means the ranking was
    rebuilt from a different table than the one GSEA ran on.
    """
    genes = [g for g in str(lead_genes).split(';') if g]
    hit = ranking.reindex(genes)
    missing = hit.isna().sum()
    assert missing == 0, f'{missing}/{len(genes)} leading-edge genes absent from the ranking'
    return hit.to_numpy(dtype=float)


def save_fig(fig, name):
    """Save a figure as SVG (600 dpi) into RIDGE_DIR. SVG only - no PNG."""
    path = os.path.join(RIDGE_DIR, name)
    fig.savefig(path, format='svg', dpi=600, bbox_inches='tight')
    print(f'  [saved] {path}')
    return path


# every KDE is evaluated on this one grid, so ridge widths are comparable across the figures
_span  = ranking.max() - ranking.min()
X_GRID = np.linspace(ranking.min() - X_PAD * _span, ranking.max() + X_PAD * _span, GRID_N)
print(f'shared x grid: {X_GRID[0]:.2f} to {X_GRID[-1]:.2f}')

## Ridge plot

One figure per database. Each row pairs the i-th strongest **Poor** term (red, left of zero) with the i-th strongest **Good** term (green, right of zero) on a shared baseline, strongest row at the top; pairing is by rank within a direction, not by any relationship between the two terms.

Each ridge is `scipy.stats.gaussian_kde` over that term's leading-edge `stat` values, evaluated on the shared `X_GRID` and peak-normalised to 1.0 — **height carries shape only**, never gene count, which is annotated next to the label instead. Fills are opaque so a lower row never shows through and reads as a spurious second mode; the white under-stroke keeps overlapping outlines separable; each ridge is clipped to its own support so its outline is not dragged flat across the other side of zero.

A direction with fewer than `TOP_N` significant terms simply yields fewer ridges on that side — rows are never padded from the other direction.

In [ ]:
def ridge_plot(top, db):
    """Draw and save one paired ridge plot.

    `top` must carry Term, nes, fdr, lead_genes and direction. It is split by direction and
    re-ranked strongest-first within each; row i then holds the i-th strongest Poor term (left
    of zero) and the i-th strongest Good term (right of zero) on a shared baseline.
    """
    poor = top[top['direction'] == 'Higher in Poor'].sort_values('nes')                   # strongest first
    good = top[top['direction'] == 'Higher in Good'].sort_values('nes', ascending=False)  # strongest first

    n_rows = max(len(poor), len(good))
    if n_rows == 0:
        print(f'  [skip] {db}: nothing plottable')
        return None

    step = 1.0 - OVERLAP

    # build the rows first, so a KDE failure can drop one side without breaking the layout
    rows = []
    for i in range(n_rows):
        entry = {'row': i + 1, 'base': (n_rows - 1 - i) * step}   # row 1 sits at the TOP
        for side, table in (('poor', poor), ('good', good)):
            entry[side] = None
            if i >= len(table):
                continue
            r = table.iloc[i]
            vals = lead_stats(r['lead_genes'], ranking)
            try:
                dens = gaussian_kde(vals)(X_GRID)
            except (LinAlgError, ValueError) as err:
                # degenerate leading edge (too few genes, or zero variance) - drop this side
                print(f'  [skip] {r["Term"]}: KDE failed ({err})')
                continue
            entry[side] = {'term': r['Term'], 'nes': float(r['nes']), 'fdr': float(r['fdr']),
                           'direction': r['direction'], 'n_lead': len(vals),
                           'median': float(np.median(vals)),
                           'lo': float(vals.min()), 'hi': float(vals.max()),
                           'curve': dens / dens.max()}
        rows.append(entry)

    fig, ax = plt.subplots(figsize=(12, 1.45 * n_rows + 2.2))

    # draw bottom row first so an upper ridge occludes a lower one (standard ridgeline depth)
    for k, entry in enumerate(sorted(rows, key=lambda e: e['base'])):
        base = entry['base']
        z = 2 * k + 2

        # one neutral baseline spanning the whole axis - it carries both ridges of the row
        ax.hlines(base, X_GRID[0], X_GRID[-1], color='0.4', linewidth=0.8, zorder=z)

        for side in ('poor', 'good'):
            d = entry[side]
            if d is None:
                continue
            color = DIR_COLORS[d['direction']]
            edge  = DIR_EDGES[d['direction']]
            y = base + d['curve']

            # Clip the ridge to its OWN support. A KDE decays to ~0 but never to exactly 0, so
            # drawing it across the full X_GRID lays a flat outline along the baseline over the
            # entire axis - which means one term's dark outline gets drawn straight across the
            # other term's ridge on the far side of zero.
            idx = np.flatnonzero(d['curve'] > SUPPORT_EPS)
            if idx.size == 0:
                continue
            lo_i, hi_i = max(idx[0] - 1, 0), min(idx[-1] + 2, len(X_GRID))
            xs, ys = X_GRID[lo_i:hi_i], y[lo_i:hi_i]

            # Fills are OPAQUE on purpose: with a translucent fill the row beneath shows through
            # the overlap and reads as a second mode in this term's density. The white
            # under-stroke keeps the overlapping outlines separable.
            ax.fill_between(xs, base, ys, facecolor=color, edgecolor='none', zorder=z)
            ax.plot(xs, ys, color='white', linewidth=3.0, solid_capstyle='round', zorder=z)
            ax.plot(xs, ys, color=edge, linewidth=1.3, solid_capstyle='round', zorder=z)

    # zero line - the pivot separating the two directions
    ax.axvline(0, color='0.25', linewidth=1.1, zorder=1)

    # ---- labels flanking the axes, on the side matching each ridge -------------------------
    # x in axes coords, y in data coords, so the blocks track their row regardless of scaling
    blend = mtransforms.blended_transform_factory(ax.transAxes, ax.transData)
    for entry in rows:
        base = entry['base']
        for side, x, ha in (('poor', -0.015, 'right'), ('good', 1.015, 'left')):
            d = entry[side]
            if d is None:
                continue
            ax.text(x, base + 0.10, clean_term(d['term']), transform=blend,
                    color=DIR_COLORS[d['direction']], fontsize=13, fontweight='bold',
                    va='bottom', ha=ha, linespacing=1.35)
            ax.text(x, base + 0.02,
                    f'NES {d["nes"]:+.2f} · {fdr_label(d["fdr"])} · '
                    f'{d["n_lead"]} lead genes',
                    transform=blend, color='0.35', fontsize=10.5, va='top', ha=ha)

    # ---- axes ------------------------------------------------------------------------------
    ax.set_yticks([])
    ax.set_xlim(X_GRID[0], X_GRID[-1])
    ax.set_ylim(-0.10, rows[0]['base'] + 1.18)
    ax.set_xlabel('Ranking metric (Wald stat)   ← Higher in Poor   │   Higher in Good →')
    ax.set_ylabel('')
    ax.set_title(f'{DB_TITLES[db]} — leading-edge gene distribution\n'
                 f'single-cell pseudobulk · top {TOP_N} terms per outcome group, '
                 f'FDR < {FDR_CUTOFF}')

    ax.grid(axis='x', color='0.92', linewidth=0.6, zorder=0)
    ax.set_axisbelow(True)
    sns.despine(ax=ax, left=True)

    present = {d['direction'] for e in rows for d in (e['poor'], e['good']) if d is not None}
    handles = [mpatches.Patch(facecolor=DIR_COLORS[d], edgecolor=DIR_EDGES[d], label=d)
               for d in ['Higher in Poor', 'Higher in Good'] if d in present]

    fig.tight_layout()
    # anchored to the figure (not the axes) so the gap is unaffected by figure height;
    # bbox_inches='tight' pulls it back into the saved SVG
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.0),
               bbox_transform=fig.transFigure, ncol=len(handles), frameon=False,
               handlelength=1.6, columnspacing=2.2)

    save_fig(fig, f'processed_gsea_{db}_ridgeplot_clinical_outcome.svg')

    tidy = pd.DataFrame([
        {'row': e['row'], 'side': side, 'term': d['term'], 'direction': d['direction'],
         'nes': d['nes'], 'fdr': d['fdr'], 'n_lead': d['n_lead'], 'median': d['median'],
         'lo': d['lo'], 'hi': d['hi']}
        for e in rows for side in ('poor', 'good') if (d := e[side]) is not None
    ])
    return fig, tidy

## Plot every enabled database

Each database's CSV is **re-read from disk** here rather than taken from the `results` dict held in memory — the plotting step is deliberately a pure function of the files on disk, so it can be re-run on its own and always plots exactly what was written. The `Unnamed: 0` index column written by `to_csv` is dropped on read.

In [ ]:
selected = {}

for db in DB_ORDER:
    csv_path = os.path.join(GSEA_OUT, CSV_BY_DB[db])
    gsea = pd.read_csv(csv_path)
    gsea = gsea.drop(columns=[c for c in gsea.columns if c.startswith('Unnamed')])

    # direction is derived from the sign of the NES - see the header cell
    gsea['direction'] = np.where(gsea['nes'] >= 0, 'Higher in Good', 'Higher in Poor')

    sig = gsea[gsea['fdr'] < FDR_CUTOFF].copy()
    n_good = int((sig['direction'] == 'Higher in Good').sum())
    n_poor = int((sig['direction'] == 'Higher in Poor').sum())

    print(f'\n########## {db} ##########')
    print(f'read {csv_path}')
    print(f'{db}: {len(gsea)} terms tested, {len(sig)} significant at fdr < {FDR_CUTOFF} '
          f'({n_good} Good / {n_poor} Poor)')

    if len(sig) == 0:
        print(f'  [skip] {db}: no terms with fdr < {FDR_CUTOFF} - nothing to plot.')
        continue

    # a direction with fewer than TOP_N significant terms simply contributes fewer ridges
    for direction, n_dir in (('Higher in Good', n_good), ('Higher in Poor', n_poor)):
        if n_dir < TOP_N:
            print(f'  [note] {direction}: only {n_dir} significant term(s) - '
                  f'fewer than the requested top {TOP_N}, plotting what is there.')

    top = select_top(sig, TOP_N)
    print(f'  -> {len(top)} ridges '
          f'({(top["direction"] == "Higher in Good").sum()} Good / '
          f'{(top["direction"] == "Higher in Poor").sum()} Poor)')

    out = ridge_plot(top, db)
    if out is not None:
        fig, tidy = out
        selected[db] = tidy
        plt.show()

## Verification

Checks that every promised artefact landed on disk with the expected schema, that each ridge sits on the side of zero its direction claims, and that the SVGs still carry live `<text>` elements — the direct proof that `svg.fonttype='none'` held and the labels stayed editable in Illustrator/Inkscape rather than being converted to paths.

The Poor-vs-Good separation is reported as a **diagnostic, not an assertion**: on a 10,000-gene HVG ranking a narrow overlap between the two leading edges of a row is a real result, not a code fault, so it is printed rather than raised.

In [ ]:
EXPECTED_COLS = ['Term', 'fdr', 'es', 'nes', 'pval',
                 'geneset_size', 'matched_size', 'tag_pct', 'lead_genes']

EXPECTED_SVGS = [f'processed_gsea_{db}_ridgeplot_clinical_outcome.svg' for db in DB_ORDER]

print('--- CSVs ---')
for db in DB_ORDER:
    path = os.path.join(GSEA_OUT, CSV_BY_DB[db])
    assert os.path.exists(path), f'missing CSV: {path}'
    chk = pd.read_csv(path)
    chk = chk.drop(columns=[c for c in chk.columns if c.startswith('Unnamed')])
    assert list(chk.columns) == EXPECTED_COLS, f'{path}: unexpected columns {list(chk.columns)}'
    print(f'  [ok] {path}  ({os.path.getsize(path) / 1024:8.1f} KB, {len(chk)} terms)')

print('\n--- ridge sign sanity ---')
bad = []
for db, tab in selected.items():
    for _, r in tab.iterrows():
        ok = (r['median'] > 0) if r['direction'] == 'Higher in Good' else (r['median'] < 0)
        if not ok:
            bad.append((db, r['term'], r['direction'], r['median']))
assert not bad, f'ridge median on the wrong side of zero: {bad}'
print(f'  [ok] all {sum(len(t) for t in selected.values())} ridges on the expected side of zero')

print('\n--- Poor/Good separation per paired row (diagnostic) ---')
margins = []
for db, tab in selected.items():
    for row, grp in tab.groupby('row'):
        sides = dict(zip(grp['side'], grp.to_dict('records')))
        if 'poor' in sides and 'good' in sides:
            margin = sides['good']['lo'] - sides['poor']['hi']
            margins.append(margin)
            flag = 'ok  ' if margin > 0 else 'OVERLAP'
            print(f'  [{flag}] {db} row {row}: gap {margin:+.2f} stat units')
if margins:
    print(f'  {len(margins)} paired row(s), narrowest gap {min(margins):+.2f} stat units')

print('\n--- SVGs ---')
for name in EXPECTED_SVGS:
    path = os.path.join(RIDGE_DIR, name)
    assert os.path.exists(path), f'missing SVG: {path}'
    with open(path) as fh:
        svg_text = fh.read()
    assert '<text' in svg_text, f'{path}: no <text> elements - svg.fonttype did not hold'
    print(f'  [ok] {path}  ({os.path.getsize(path) / 1024:8.1f} KB, text editable)')

pngs = [f for f in os.listdir(RIDGE_DIR) if f.endswith('.png')]
assert not pngs, f'unexpected PNGs in {RIDGE_DIR}: {pngs}'

print(f'\nAll {len(DB_ORDER)} CSVs and {len(EXPECTED_SVGS)} ridge SVGs verified.')

In [ ]:
# what actually got plotted, for the record
pd.concat(selected, names=['db']).reset_index(level=0)[
    ['db', 'row', 'side', 'term', 'direction', 'nes', 'fdr', 'n_lead', 'median']
].reset_index(drop=True)